# Anatomía de una canción popular — Spotify Tracks Dataset

**Universidad de Playa Ancha · Ingeniería en Informática · Primer Semestre 2026**  
**Asignatura:** Ciencia de Datos con Python (Optativo II) — Prof. Diego Saavedra Lizana

**Integrantes:** Martín Maturana · Vicente Tapia · Juan Pablo Poblete · Moisés Espinoza · Benjamín Burgos

**Pregunta de investigación:**
> ¿Qué características de audio (danceability, energy, valence, tempo) predicen mejor la popularidad de una canción en Spotify, y varía este patrón según el género musical?

📄 [Análisis completo e interpretaciones](../docs/analisis.md)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
import os

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
os.makedirs('../figures', exist_ok=True)

## 1 — Carga y limpieza

In [ ]:
df = pd.read_csv('../data/dataset.csv')

# Eliminar columnas irrelevantes (índice del CSV, ID interno, nombre de álbum)
df = df.drop(columns=['Unnamed: 0', 'track_id', 'album_name'], errors='ignore')

# Eliminar nulos y duplicados
df = df.dropna().drop_duplicates().reset_index(drop=True)

# Transformaciones
df['explicit']     = df['explicit'].astype(int)
df['duration_min'] = df['duration_ms'] / 60000

# Categoría de popularidad
def categorizar_popularidad(p):
    if p <= 25:   return 'emerging'
    elif p <= 50: return 'up_and_coming'
    elif p <= 75: return 'mainstream'
    else:         return 'chart_topper'

orden_cat   = ['emerging', 'up_and_coming', 'mainstream', 'chart_topper']
palette_cat = {'emerging': '#d62728', 'up_and_coming': '#ff7f0e',
               'mainstream': '#2ca02c', 'chart_topper': '#1f77b4'}

df['popularity_cat'] = pd.Categorical(
    df['popularity'].apply(categorizar_popularidad),
    categories=orden_cat, ordered=True
)

# Variables de audio
vars_audio = ['danceability', 'energy', 'loudness', 'speechiness',
              'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

df_clean = df.copy()
print(f"Dataset listo: {df_clean.shape[0]:,} canciones · {df_clean.shape[1]} columnas")
df_clean.head(3)

## 2 — Distribución de popularidad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df_clean['popularity'], kde=True, ax=axes[0], color='steelblue', bins=40)
axes[0].set_title('Distribución de popularidad')
axes[0].set_xlabel('Popularidad (0–100)')
axes[0].set_ylabel('Frecuencia')

sns.boxplot(x=df_clean['popularity'], ax=axes[1], color='steelblue')
axes[1].set_title('Boxplot de popularidad')
axes[1].set_xlabel('Popularidad (0–100)')

plt.tight_layout()
plt.savefig('../figures/01_popularity_distribucion.png', bbox_inches='tight')
plt.show()

print(f"Media: {df_clean['popularity'].mean():.2f} | Mediana: {df_clean['popularity'].median():.2f} | Skewness: {df_clean['popularity'].skew():.4f}")

## 3 — Predictores de audio

In [ ]:
vars_numericas = vars_audio + ['duration_min', 'explicit', 'key', 'mode', 'time_signature']
correlaciones  = df_clean[vars_numericas].corrwith(df_clean['popularity']).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['coral' if c < 0 else 'steelblue' for c in correlaciones.values]
ax.barh(correlaciones.index, correlaciones.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de Pearson con popularidad')
ax.set_xlabel('Coeficiente de correlación')
plt.tight_layout()
plt.savefig('../figures/02_correlacion_popularidad.png', bbox_inches='tight')
plt.show()

print("Top 5 positivas:"); print(correlaciones.tail(5).to_string())
print("\nTop 5 negativas:"); print(correlaciones.head(5).to_string())

In [ ]:
vars_pregunta = ['danceability', 'energy', 'valence', 'tempo']
muestra = df_clean.sample(10000, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, var in enumerate(vars_pregunta):
    for cat in orden_cat:
        sub = muestra[muestra['popularity_cat'] == cat]
        axes[i].scatter(sub[var], sub['popularity'],
                        alpha=0.3, s=8, label=cat, color=palette_cat[cat])
    m, b = np.polyfit(df_clean[var], df_clean['popularity'], 1)
    x_line = np.linspace(df_clean[var].min(), df_clean[var].max(), 100)
    axes[i].plot(x_line, m*x_line + b, color='black', linewidth=1.5)
    axes[i].set_title(f'{var} vs popularidad  (r={correlaciones[var]:.3f})')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Popularidad')
    if i == 0:
        axes[i].legend(title='Categoría', fontsize=7)

plt.suptitle('Variables de la pregunta de investigación vs popularidad', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/03_vars_pregunta_vs_popularidad.png', bbox_inches='tight')
plt.show()

## 4 — Análisis por género

In [ ]:
top10_generos = df_clean['track_genre'].value_counts().head(10).index.tolist()
df_top10      = df_clean[df_clean['track_genre'].isin(top10_generos)]
orden_genero  = df_top10.groupby('track_genre')['popularity'].median().sort_values().index

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_top10, x='popularity', y='track_genre',
            order=orden_genero, palette='viridis', ax=ax)
ax.set_title('Popularidad por género musical (top 10)')
ax.set_xlabel('Popularidad')
ax.set_ylabel('Género')
plt.tight_layout()
plt.savefig('../figures/04_popularidad_por_genero.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, var in zip(axes, ['danceability', 'energy', 'valence']):
    sns.boxplot(data=df_top10, x=var, y='track_genre',
                order=orden_genero, palette='coolwarm', ax=ax)
    ax.set_title(f'{var} por género')
    ax.set_xlabel(var)
    ax.set_ylabel('Género')

plt.suptitle('Características de audio por género (top 10)', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/05_audio_por_genero.png', bbox_inches='tight')
plt.show()

## 5 — Modelo predictivo

In [ ]:
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness',
            'valence', 'tempo', 'explicit', 'duration_min',
            'key', 'mode', 'time_signature']

X = df_clean[features]
y = df_clean['popularity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# Entrenar modelos
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# Métricas
def evaluar_modelo(nombre, y_test, y_pred):
    return {'Modelo': nombre,
            'R²':   round(r2_score(y_test, y_pred), 4),
            'MAE':  round(mean_absolute_error(y_test, y_pred), 2),
            'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 2)}

resultados = pd.DataFrame([
    evaluar_modelo('Regresión Lineal',  y_test, y_pred_lr),
    evaluar_modelo('Árbol de Decisión', y_test, y_pred_dt),
]).set_index('Modelo')
print(resultados)

In [ ]:
fi_df = pd.DataFrame({'feature': features, 'importancia': dt.feature_importances_})
fi_df = fi_df.sort_values('importancia', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(fi_df['feature'], fi_df['importancia'], color='mediumseagreen')
ax.set_title('Feature importance — Árbol de Decisión (max_depth=5)')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.savefig('../figures/06_feature_importance_dt.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (nombre, y_pred) in zip(axes, [('Regresión Lineal', y_pred_lr),
                                        ('Árbol de Decisión', y_pred_dt)]):
    ax.scatter(y_test, y_pred, alpha=0.2, s=5, color='steelblue')
    lims = [0, 100]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
    ax.set_title(f'{nombre}\nR²={r2_score(y_test, y_pred):.4f}')
    ax.set_xlabel('Popularidad real')
    ax.set_ylabel('Popularidad predicha')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend()

plt.tight_layout()
plt.savefig('../figures/07_prediccion_vs_real.png', bbox_inches='tight')
plt.show()